# Synthetic CMAT quick-start

This notebook is the lightest notebook-form CMAT example currently maintained in the rebuild. It mirrors `examples/synthetic_ttv_quickstart.py` and avoids remote TESS access, transit fitting, and the full production grid.

## What it covers

- define a small synthetic TTV series
- run a reduced companion grid with `cmat.ttv_sim`
- compute reduced critical-mass outputs
- evaluate one reduced MEGNO point

If your environment needs writable cache directories for Matplotlib-related imports, launch Jupyter with the same `MPLCONFIGDIR` and `XDG_CACHE_HOME` settings described in `docs/TROUBLESHOOTING.md`.

In [ ]:
import numpy as np

from cmat.ttv_sim import ttv_sim

In [ ]:
epochs = np.array([0, 1, 2, 3])
ttv_mcmc = np.array([12.0, -8.0, 15.0, -10.0])
ttv_err = np.full(4, 5.0)
prop = [
    {
        "orbital_distance": 0.055,
        "orbital_period": 4.0,
        "Mp": 1.0,
        "Ms": 1.0,
        "Rs": 1.0,
        "Rp": 1.0,
    }
]
period_ratios = np.array([1.5, 2.0])
companion_masses = np.array([10.0, 20.0])

In [ ]:
simulation = ttv_sim(
    epochs=epochs,
    ttv_mcmc=ttv_mcmc,
    ttv_err=ttv_err,
    rs=period_ratios,
    mp2s=companion_masses,
    prop=prop,
)

simulation.ttv_results = [
    simulation.calculate_rebound((period_ratio, companion_mass))
    for companion_mass in simulation.mp2s
    for period_ratio in simulation.rs
]
chi2_limit, rms_limit = simulation.get_m_crit()

print("chi2 limits:", chi2_limit.tolist())
print("rms limits:", rms_limit.tolist())

In [ ]:
simulation.megno_dt = 0.02
simulation.megno_runtime = 50.0
megno = simulation.simulation_m((1.5, 10.0))

print("sample megno:", float(megno))

## Interpreting the output

- Empty reduced mass-limit arrays mean the tiny validation grid did not cross the current rejection threshold.
- The MEGNO value here is just a single reduced diagnostic point, not a full stability map.
- For the end-to-end astronomy workflow, use `example.ipynb` instead.